# QB Value Analysis: Does Defense Win Championships?

## Hypothesis

Tom Brady's status as the "GOAT" conflates team success with individual QB value. The Patriots consistently fielded **top-10 defenses**, and the core claim here is that **top defenses reliably produce 10+ win seasons regardless of QB quality**.

We further hypothesize that **Peyton Manning and Aaron Rodgers were the best pure QBs ever**, because they maintained elite passer ratings even when their defenses provided little support.

Conversely, several Super Bowl championships were won by **clipboard QBs** (Dilfer, McMahon, Williams, Johnson) riding elite defenses.

**Data**: NFL-only passing stats 1960–2024 (`/Users/devos/data/pfref/`). Pre-1970 AFL seasons (e.g., Namath's 1968 SB III) are absent from this dataset.

In [1]:
import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

try:
    import ipywidgets as widgets
    import anywidget  # required by plotly FigureWidget >= 5.18
    from plotly.graph_objects import FigureWidget
    from IPython.display import display
    WIDGETS_OK = True
except ImportError as _e:
    WIDGETS_OK = False
    print(f'Widget libraries not fully available ({_e}) — linked charts will use subplot mode instead')

## NFL Passer Rating Formula

Four components, each clamped to `[0, 2.375]`, then combined:

$$a = \left(\frac{\text{Comp}}{\text{Att}} - 0.30\right) \times 5 \qquad b = \left(\frac{\text{Yds}}{\text{Att}} - 3\right) \times 0.25$$

$$c = \frac{\text{TD}}{\text{Att}} \times 20 \qquad d = 2.375 - \frac{\text{Int}}{\text{Att}} \times 25$$

$$\text{Passer Rating} = \frac{a+b+c+d}{6} \times 100 \quad \text{(max = 158.3)}$$

> Source: [Wikipedia — Passer rating](https://en.wikipedia.org/wiki/Passer_rating)

In [2]:
def nfl_passer_rating(comp, att, yards, td, ints):
    """Official NFL passer rating from season totals. Returns NaN when att == 0."""
    if att == 0:
        return np.nan
    def clamp(x): return max(0.0, min(2.375, x))
    a = clamp((comp / att - 0.30) * 5)
    b = clamp((yards / att - 3.0) * 0.25)
    c = clamp((td / att) * 20)
    d = clamp(2.375 - (ints / att * 25))
    return round((a + b + c + d) / 6 * 100, 1)

assert nfl_passer_rating(362, 564, 5084, 48, 17) == 108.9, 'Marino 1984'
assert nfl_passer_rating(398, 578, 4806, 50,  8) == 117.2, 'Brady 2007'
assert nfl_passer_rating(134, 226, 1502, 12, 11) ==  76.6, 'Dilfer 2000'
print('Passer rating formula verified.')

Passer rating formula verified.


In [3]:
def abbrev_to_team_name(abbrev, year):
    """Map PFR abbreviation + year to franchise name used in team-history CSVs."""
    if abbrev in ('2TM', 'BCL', 'NYY', None):
        return None
    year_dep = {
        'STL': lambda y: 'St. Louis Cardinals' if y <= 1987 else 'St. Louis Rams',
        'BAL': lambda y: 'Baltimore Colts' if y <= 1983 else 'Baltimore Ravens',
        'HOU': lambda y: 'Houston Oilers' if y <= 1996 else ('Tennessee Oilers' if y <= 1998 else 'Houston Texans'),
        'TEN': lambda y: 'Tennessee Oilers' if y <= 1998 else 'Tennessee Titans',
        'OAK': lambda y: 'Oakland Raiders',
        'RAI': lambda y: 'Los Angeles Raiders',
        'LVR': lambda y: 'Las Vegas Raiders',
        'RAM': lambda y: 'Los Angeles Rams',
        'LAR': lambda y: 'Los Angeles Rams',
        'SDG': lambda y: 'San Diego Chargers',
        'LAC': lambda y: 'Los Angeles Chargers',
        'PHO': lambda y: 'Phoenix Cardinals',
        'ARI': lambda y: 'Arizona Cardinals',
        'CRD': lambda y: 'Chicago Cardinals',
        'BOS': lambda y: 'Boston Patriots',
        'NWE': lambda y: 'New England Patriots',
        'WAS': lambda y: 'Washington Redskins' if y <= 2019 else ('Washington Football Team' if y <= 2021 else 'Washington Commanders'),
        'DTX': lambda y: 'Dallas Texans',
    }
    simple = {
        'ATL': 'Atlanta Falcons', 'BUF': 'Buffalo Bills', 'CAR': 'Carolina Panthers',
        'CHI': 'Chicago Bears', 'CIN': 'Cincinnati Bengals', 'CLE': 'Cleveland Browns',
        'DAL': 'Dallas Cowboys', 'DEN': 'Denver Broncos', 'DET': 'Detroit Lions',
        'GNB': 'Green Bay Packers', 'IND': 'Indianapolis Colts', 'JAX': 'Jacksonville Jaguars',
        'KAN': 'Kansas City Chiefs', 'MIA': 'Miami Dolphins', 'MIN': 'Minnesota Vikings',
        'NOR': 'New Orleans Saints', 'NYG': 'New York Giants', 'NYJ': 'New York Jets',
        'PHI': 'Philadelphia Eagles', 'PIT': 'Pittsburgh Steelers',
        'SFO': 'San Francisco 49ers', 'SEA': 'Seattle Seahawks', 'TAM': 'Tampa Bay Buccaneers',
    }
    return year_dep[abbrev](year) if abbrev in year_dep else simple.get(abbrev)

## Load & Build Dataset

In [4]:
PASSING_DIR = '/Users/devos/data/pfref/player-stats/passing/'
TEAM_HIST_DIR = '/Users/devos/data/pfref/team-history/'

frames = []
for path in sorted(glob.glob(f'{PASSING_DIR}passing_*.csv')):
    year = int(path.split('_')[-1].replace('.csv', ''))
    df = pd.read_csv(path, low_memory=False)
    df['year'] = year
    frames.append(df)
passing_raw = pd.concat(frames, ignore_index=True)
for col in ['comp', 'att', 'yards', 'td', 'int', 'games_started', 'qb_rating']:
    if col in passing_raw.columns:
        passing_raw[col] = pd.to_numeric(passing_raw[col], errors='coerce')
passing_raw = passing_raw[(passing_raw['att'].fillna(0) > 0) & (passing_raw['team_abbrev'] != '2TM')].copy()

th_frames = [pd.read_csv(p, low_memory=False) for p in glob.glob(f'{TEAM_HIST_DIR}*.csv')]
team_hist = pd.concat(th_frames, ignore_index=True)
for col in ['wins', 'losses', 'ties', 'def_pts_rank', 'def_yds_rank', 'number_teams']:
    team_hist[col] = pd.to_numeric(team_hist[col], errors='coerce')
team_hist = team_hist[team_hist['year'] >= 1960].copy()

print(f'Passing rows: {len(passing_raw):,}  ({passing_raw.year.min()}–{passing_raw.year.max()})')
print(f'Team-history rows: {len(team_hist):,}')

Passing rows: 6,767  (1950–2024)
Team-history rows: 1,869


In [5]:
def aggregate_team_qbs(df_year, team_abbrev, year):
    """Aggregate all passing attempts for one team-season → composite passer rating."""
    qbs = df_year[(df_year['team_abbrev'] == team_abbrev) & (df_year['att'].fillna(0) > 0)].copy()
    if qbs.empty:
        return None
    totals = qbs[['comp', 'att', 'yards', 'td', 'int']].sum()
    # Primary QB = player with most passing attempts.
    # games_started reflects total starts (any position), so a RB who starts 16 games would
    # rank above a QB with 9 starts (e.g., Walter Payton vs Jim McMahon 1984).
    row = qbs.loc[qbs['att'].idxmax()]
    return pd.Series({
        'year': year, 'team_abbrev': team_abbrev,
        'primary_qb': row['player_name'],
        'primary_qb_id': row['player_id'],
        'primary_starts': row.get('games_started', 0),
        'primary_rating': float(row['qb_rating']) if pd.notna(row.get('qb_rating')) else np.nan,
        'num_qbs': len(qbs),
        'composite_rating': nfl_passer_rating(totals['comp'], totals['att'], totals['yards'], totals['td'], totals['int']),
        'total_att': totals['att'], 'total_comp': totals['comp'],
        'total_yards': totals['yards'], 'total_td': totals['td'], 'total_int': totals['int'],
    })

records = []
for year in sorted(p for p in passing_raw['year'].unique() if p >= 1960):
    df_y = passing_raw[passing_raw['year'] == year]
    for ta in df_y['team_abbrev'].unique():
        if ta in ('2TM', None) or pd.isna(ta): continue
        agg = aggregate_team_qbs(df_y, ta, year)
        if agg is not None:
            agg['team_name'] = abbrev_to_team_name(ta, year)
            records.append(agg)

team_qb = pd.DataFrame(records)
team_qb = team_qb[team_qb['team_name'].notna()].copy()

joined = team_qb.merge(
    team_hist[['year', 'team_name', 'wins', 'losses', 'ties',
               'def_pts_rank', 'def_yds_rank', 'number_teams', 'playoffs']],
    on=['year', 'team_name'], how='left'
)
joined['def_pts_norm'] = (joined['def_pts_rank'] - 1) / (joined['number_teams'] - 1)
joined['def_yds_norm'] = (joined['def_yds_rank'] - 1) / (joined['number_teams'] - 1)

def classify_sb(p):
    if pd.isna(p) or str(p).strip() == '': return 'No Playoffs'
    if 'Won SB' in str(p): return 'Won Super Bowl'
    if 'Lost SB' in str(p): return 'Lost Super Bowl'
    return 'Playoffs (no SB)'
joined['sb_result'] = joined['playoffs'].apply(classify_sb)

# Season length varies: 12 games pre-1961, 14 games 1961-1977, 16 games 1978-2020, 17 games 2021+
def season_games(year):
    if year <= 1960: return 12
    if year <= 1977: return 14
    if year <= 2020: return 16
    return 17
joined['season_games'] = joined['year'].apply(season_games)
joined['win_pct'] = joined['wins'] / joined['season_games']

print(f'Joined dataset: {len(joined):,} team-seasons | SB winners: {(joined.sb_result=="Won Super Bowl").sum()}')

Joined dataset: 1,783 team-seasons | SB winners: 57


## Test Case: 1984–1986 Bears

In [6]:
for year in [1984, 1985, 1986]:
    df_y = passing_raw[passing_raw['year'] == year]
    chi = df_y[(df_y['team_abbrev'] == 'CHI') & (df_y['att'].fillna(0) > 0)]
    totals = chi[['comp','att','yards','td','int']].sum()
    comp_r = nfl_passer_rating(totals['comp'],totals['att'],totals['yards'],totals['td'],totals['int'])
    starters = chi[chi['games_started'].fillna(0) > 0].sort_values('games_started', ascending=False)
    th_row = joined[(joined['year']==year) & (joined['team_name']=='Chicago Bears')].iloc[0]
    print(f'{year} Bears — {int(th_row.wins)}-{int(th_row.losses)} — def pts rank {int(th_row.def_pts_rank)}/{int(th_row.number_teams)} — composite QB rating: {comp_r}')
    for _, r in starters.iterrows():
        print(f'  {r.player_name}: {int(r.games_started)} starts, {r.qb_rating} rating')

1984 Bears — 10-6 — def pts rank 3/28 — composite QB rating: 75.1
  Walter Payton: 16 starts, 57.8 rating
  Matt Suhey: 16 starts, 39.6 rating
  Jim McMahon: 9 starts, 97.8 rating
  Steve Fuller: 4 starts, 103.3 rating
  Rusty Lisch: 1 starts, 35.1 rating
  Bob Avellini: 1 starts, 48.3 rating
  Greg Landry: 1 starts, 66.5 rating
  Brian Baschnagel: 1 starts, 58.3 rating
1985 Bears — 15-1 — def pts rank 1/28 — composite QB rating: 77.3
  Walter Payton: 16 starts, 143.7 rating
  Jim McMahon: 11 starts, 82.6 rating
  Steve Fuller: 5 starts, 57.3 rating
1986 Bears — 14-2 — def pts rank 1/28 — composite QB rating: 57.6
  Walter Payton: 16 starts, 0.0 rating
  Mike Tomczak: 7 starts, 50.2 rating
  Jim McMahon: 6 starts, 61.4 rating
  Steve Fuller: 2 starts, 60.1 rating
  Doug Flutie: 1 starts, 80.1 rating


---
# Chart 1 — Win% Distribution by Defensive Tier

Split all team-seasons into five defensive tiers and show the win% distribution for each. The tier labels show what % of teams in that tier reached 10+ wins.

In [7]:
TIER_CUTS = [0, 0.10, 0.25, 0.50, 0.75, 1.01]
TIER_LABELS = ['Top 10%\n(Elite)', '11–25%\n(Very Good)', '26–50%\n(Average)', '51–75%\n(Below Avg)', 'Bottom 25%\n(Poor)']
TIER_COLORS = ['#1a9850', '#66bd63', '#fdae61', '#f46d43', '#d73027']

plot_tier = joined[joined['def_pts_norm'].notna() & joined['win_pct'].notna()].copy()
plot_tier['tier'] = pd.cut(plot_tier['def_pts_norm'], bins=TIER_CUTS, labels=TIER_LABELS, right=False)

fig_dist = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Win% Distribution by Defensive Tier', 'P(10+ win pace) per Tier'],
    column_widths=[0.65, 0.35],
)

for i, (tier, color) in enumerate(zip(TIER_LABELS, TIER_COLORS)):
    subset = plot_tier[plot_tier['tier'] == tier]['win_pct'].dropna()
    p10 = (subset >= 0.625).mean()
    hover_v = [f'{tier.replace(chr(10)," ")} — {pct:.1%}' for pct in subset]
    fig_dist.add_trace(go.Violin(
        x=[tier.replace('\n', ' ')] * len(subset),
        y=subset,
        box_visible=True,
        meanline_visible=True,
        fillcolor=color,
        opacity=0.7,
        line_color='black',
        name=tier.replace('\n', ' '),
        hovertext=hover_v, hoverinfo='text',
        showlegend=False,
    ), row=1, col=1)

    # Bar for p10
    fig_dist.add_trace(go.Bar(
        x=[tier.replace('\n', ' ')],
        y=[p10],
        marker_color=color,
        text=[f'{p10:.0%}'],
        textposition='outside',
        showlegend=False,
        name=tier.replace('\n', ' '),
        hovertemplate=f'{tier.replace(chr(10)," ")}<br>P(10+ wins): {p10:.1%}<extra></extra>',
    ), row=1, col=2)

fig_dist.add_hline(y=0.625, line_dash='dot', line_color='navy', line_width=2,
                   annotation_text='10-win pace', annotation_position='right',
                   row=1, col=1)

fig_dist.update_yaxes(title_text='Win%', tickformat='.0%', row=1, col=1)
fig_dist.update_yaxes(title_text='P(10+ win pace)', tickformat='.0%', row=1, col=2)
fig_dist.update_layout(
    title=dict(text='Win% Distribution by Defensive Tier', x=0.5),
    height=500,
    hovermode='closest',
)
fig_dist.show()

print('\nTier breakdown:')
for tier in TIER_LABELS:
    s = plot_tier[plot_tier['tier'] == tier]
    print(f'  {tier.replace(chr(10)," "):22s}  n={len(s):4d}  avg win%={s.win_pct.mean():.1%}  '
          f'10+ wins={( s.win_pct>=0.625).mean():.0%}  '
          f'Won SB={(s.sb_result=="Won Super Bowl").mean():.1%}')


Tier breakdown:
  Top 10% (Elite)         n= 211  avg win%=71.0%  10+ wins=83%  Won SB=14.2%
  11–25% (Very Good)      n= 247  avg win%=63.0%  10+ wins=62%  Won SB=6.1%
  26–50% (Average)        n= 437  avg win%=54.3%  10+ wins=36%  Won SB=1.8%
  51–75% (Below Avg)      n= 435  avg win%=44.7%  10+ wins=16%  Won SB=0.7%
  Bottom 25% (Poor)       n= 453  avg win%=30.9%  10+ wins=3%  Won SB=0.2%


---
# Chart 2 — All Super Bowl Appearances

Every team that appeared in a Super Bowl (1970–2024) is plotted here. The primary regular-season QB (most starts) for that team is labeled.
- **Gold star** = Won the Super Bowl
- **Blue diamond** = Lost the Super Bowl
- X-axis = how good was the defense (left = elite, right = weak)
- Y-axis = composite QB passer rating

**Bottom-left quadrant** = great defense + mediocre QB → still won. **Top-right** = elite QB with poor defense.

In [8]:
sb_all = joined[
    joined['sb_result'].isin(['Won Super Bowl', 'Lost Super Bowl']) &
    joined['def_pts_norm'].notna() &
    joined['composite_rating'].notna()
].copy()

sb_all['label'] = sb_all['primary_qb'].str.split().str[-1] + "'" + sb_all['year'].astype(str).str[-2:]
sb_all['hover_sb'] = (
    '<b>' + sb_all['primary_qb'] + '</b> — ' + sb_all['year'].astype(str) +
    '<br>Team: ' + sb_all['team_name'] +
    '<br>QB Rating: ' + sb_all['composite_rating'].round(1).astype(str) +
    '<br>Def Pts Rank: ' + sb_all['def_pts_rank'].astype('Int64').astype(str) +
    ' / ' + sb_all['number_teams'].astype('Int64').astype(str) +
    '  (norm: ' + sb_all['def_pts_norm'].round(2).astype(str) + ')' +
    '<br>Record: ' + sb_all['wins'].astype(int).astype(str) +
    '-' + sb_all['losses'].astype(int).astype(str) +
    '<br>' + sb_all['sb_result']
)

fig_sb = go.Figure()

for outcome, sym, color, opacity in [
    ('Won Super Bowl',  'star',    '#FFD700', 0.9),
    ('Lost Super Bowl', 'diamond', '#4a9ede', 0.75),
]:
    s = sb_all[sb_all['sb_result'] == outcome]
    fig_sb.add_trace(go.Scatter(
        x=s['def_pts_norm'],
        y=s['composite_rating'],
        mode='markers+text',
        marker=dict(symbol=sym, size=14, color=color, opacity=opacity,
                    line=dict(width=1.2, color='black')),
        text=s['label'],
        textposition='top center',
        textfont=dict(size=8),
        name=outcome,
        hovertext=s['hover_sb'],
        hoverinfo='text',
    ))

# Quadrant shading
for x0, x1, y0, y1, fc, text in [
    (0,    0.35, 0,  85,  'rgba(255,200,0,0.06)',  'Elite D + mediocre QB'),
    (0.35, 1.0,  100, 160, 'rgba(70,130,180,0.06)', 'Great QB + weak D'),
]:
    fig_sb.add_shape(type='rect', x0=x0, x1=x1, y0=y0, y1=y1,
                     fillcolor=fc, line_width=0)
    fig_sb.add_annotation(x=(x0+x1)/2, y=y1-4, text=text,
                           showarrow=False, font=dict(size=9, color='gray'))

# Reference lines
fig_sb.add_hline(y=85, line_dash='dot', line_color='gray', line_width=1.5,
                 annotation_text='85 rating threshold', annotation_position='left')
fig_sb.add_vline(x=0.35, line_dash='dot', line_color='gray', line_width=1.5,
                 annotation_text='Top-35% defense', annotation_position='top')

fig_sb.update_layout(
    title=dict(text='All Super Bowl Appearances: QB Rating vs. Defensive Support (1970–2024)', x=0.5),
    xaxis=dict(title='← Best Defense    Worst Defense →', tickformat='.0%', range=[-0.04, 1.04]),
    yaxis=dict(title='Composite QB Passer Rating', range=[38, 160]),
    height=640,
    hovermode='closest',
    legend=dict(orientation='h', y=-0.1),
)
fig_sb.show()

print(f'SB appearances plotted: {len(sb_all)} ({sb_all.sb_result.value_counts().to_dict()})')

SB appearances plotted: 114 ({'Won Super Bowl': 57, 'Lost Super Bowl': 57})


---
# Chart 3 — Top QB Seasons: Win% vs. Defense PPG Rank: Win% vs. Defense PPG Rank

Every season for each QB in our list plotted as a dot.  
**X-axis** = normalized defense PPG rank (0 = league's best defense, 1 = worst).  
**Y-axis** = season win% (normalized for 12/14/16/17 game schedules).  
**Gray dots** = all other team-seasons as context. **Dashed curve** = league-average win% at each defense tier.

The story: dots cluster left when defense is elite — and win% is high regardless of who the QB was.

In [9]:
GOAT_IDS = {
    'StarBa00': 'Bart Starr',     'UnitJo00': 'Johnny Unitas',
    'DawsLe00': 'Len Dawson',     'StauRo00': 'Roger Staubach',
    'GrieBo00': 'Bob Griese',     'TarkFr00': 'Fran Tarkenton',
    'BradTe00': 'Terry Bradshaw', 'MontJo01': 'Joe Montana',
    'MariDa00': 'Dan Marino',     'ElwaJo00': 'John Elway',
    'FavrBr00': 'Brett Favre',    'MannPe00': 'Peyton Manning',
    'BradTo00': 'Tom Brady',      'RodgAa00': 'Aaron Rodgers',
}
CLIPBOARD_IDS = {
    'DilfTr00': 'Trent Dilfer', 'JohnBr00': 'Brad Johnson',
    'McMaJi00': 'Jim McMahon',  'SimmPh00': 'Phil Simms',
    'WillDo01': 'Doug Williams','NamaJo00': 'Joe Namath',
}
TOP_QB_IDS = {**GOAT_IDS, **CLIPBOARD_IDS}

tgt = joined[
    joined['primary_qb_id'].isin(TOP_QB_IDS) &
    joined['def_pts_norm'].notna() &
    joined['win_pct'].notna()
].copy()

# ── Background: all team-seasons ──────────────────────────────────────────────
bg = joined[joined['def_pts_norm'].notna() & joined['win_pct'].notna()].copy()
smooth_bg = bg.sort_values('def_pts_norm').reset_index(drop=True)
w_bg = max(50, int(len(smooth_bg) * 0.07))
smooth_bg['roll'] = smooth_bg['win_pct'].rolling(w_bg, center=True, min_periods=w_bg // 3).mean()
smooth_bg = smooth_bg.dropna(subset=['roll'])

palette = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24
qb_sorted = sorted(tgt['primary_qb'].unique())
cmap = {qb: palette[i % len(palette)] for i, qb in enumerate(qb_sorted)}

fig_topqb = go.Figure()

fig_topqb.add_trace(go.Scatter(
    x=bg['def_pts_norm'], y=bg['win_pct'],
    mode='markers',
    marker=dict(color='rgba(200,200,200,0.18)', size=3.5),
    name='All teams', hoverinfo='skip', showlegend=False,
))
fig_topqb.add_trace(go.Scatter(
    x=smooth_bg['def_pts_norm'], y=smooth_bg['roll'],
    mode='lines',
    line=dict(color='rgba(80,80,80,0.50)', width=2.5, dash='dot'),
    name='League avg', hoverinfo='skip',
))

for qb in qb_sorted:
    sub = tgt[tgt['primary_qb'] == qb]
    hover = (
        '<b>' + qb + '</b> — ' + sub['year'].astype(str) +
        '<br>Team: ' + sub['team_name'] +
        '<br>Record: ' + sub['wins'].astype(int).astype(str) + '-' +
                         sub['losses'].astype(int).astype(str) +
        '  (Win% ' + (sub['win_pct'] * 100).round(1).astype(str) + '%)' +
        '<br>Def PPG Rank: ' + sub['def_pts_rank'].astype('Int64').astype(str) +
        '/' + sub['number_teams'].astype('Int64').astype(str) +
        '<br>QB Rating: ' + sub['composite_rating'].round(1).astype(str) +
        '<br>' + sub['sb_result']
    )
    fig_topqb.add_trace(go.Scatter(
        x=sub['def_pts_norm'], y=sub['win_pct'],
        mode='markers',
        marker=dict(color=cmap[qb], size=7, opacity=0.85,
                    line=dict(width=0.8, color='white')),
        name=qb, hovertext=hover, hoverinfo='text',
    ))

fig_topqb.add_hline(y=0.625, line_dash='dot', line_color='#1a7a1a', line_width=2,
                    annotation_text='10-win equivalent (62.5%)', annotation_position='right',
                    annotation_font_size=11)
fig_topqb.add_hline(y=0.500, line_dash='dot', line_color='#888', line_width=1,
                    annotation_text='.500 pace', annotation_position='right',
                    annotation_font_size=10)
fig_topqb.add_vline(x=0.25, line_dash='dot', line_color='#bbb', line_width=1,
                    annotation_text='Top-25% def', annotation_position='top',
                    annotation_font_size=9)
fig_topqb.add_vline(x=0.75, line_dash='dot', line_color='#bbb', line_width=1,
                    annotation_text='Bottom-25% def', annotation_position='top',
                    annotation_font_size=9)

fig_topqb.update_layout(
    title=dict(text='All Top QB Seasons: Win% vs. Team Defense PPG Rank<br>'
                    '<sup>Each dot = one regular season. Dotted gray curve = league avg at each defense tier.</sup>',
               x=0.5),
    xaxis=dict(title='← Elite Defense (Best PPG)     Weak Defense (Worst PPG) →',
               tickformat='.0%', range=[-0.03, 1.03]),
    yaxis=dict(title='Normalized Win%', tickformat='.0%', range=[-0.03, 1.05]),
    height=600, hovermode='closest',
    legend=dict(orientation='v', x=1.01, y=1,
                bgcolor='rgba(255,255,255,0.85)',
                bordercolor='lightgray', borderwidth=1),
)
fig_topqb.show()


---
# Chart 4 — Correlation: QB Rating and Defense vs. Wins: What Actually Predicts Wins?

Two simple dot plots across all 1,700+ team-seasons.  
**Left**: QB composite passer rating vs win% — shows a *weak* positive relationship. A great offense doesn't guarantee wins.  
**Right**: Team defense PPG rank vs win% — a much **stronger** relationship.  

The r values quantify this directly.

In [10]:
from scipy import stats as scipy_stats

fig_corr = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Composite QB Rating vs Win%', 'Defense PPG Rank vs Win%'],
    horizontal_spacing=0.12,
)

# ── Left: QB Rating vs Win% ────────────────────────────────────────────────
df_qb = joined[joined['composite_rating'].notna() & joined['win_pct'].notna()].copy()
r_qb, _ = scipy_stats.pearsonr(df_qb['composite_rating'], df_qb['win_pct'])
m_qb, b_qb = np.polyfit(df_qb['composite_rating'], df_qb['win_pct'], 1)
x_qb = np.linspace(df_qb['composite_rating'].min(), df_qb['composite_rating'].max(), 300)

hover_qb = (
    df_qb['year'].astype(str) + ' ' + df_qb['team_name'] +
    '<br>QB: ' + df_qb['primary_qb'].fillna('?') +
    '<br>Rating: ' + df_qb['composite_rating'].round(1).astype(str) +
    '<br>Win%: ' + (df_qb['win_pct'] * 100).round(1).astype(str) + '%'
)
fig_corr.add_trace(go.Scatter(
    x=df_qb['composite_rating'], y=df_qb['win_pct'],
    mode='markers',
    marker=dict(color='steelblue', size=3, opacity=0.30),
    hovertext=hover_qb, hoverinfo='text',
    name='All team-seasons', showlegend=False,
), row=1, col=1)
fig_corr.add_trace(go.Scatter(
    x=x_qb, y=m_qb * x_qb + b_qb,
    mode='lines', line=dict(color='navy', width=2.5),
    name='Trend', hoverinfo='skip', showlegend=False,
), row=1, col=1)
fig_corr.add_annotation(
    row=1, col=1, x=0.05, y=0.92, xref='x domain', yref='y domain',
    text=f'<b>r = {r_qb:.3f}</b>',
    showarrow=False, font=dict(size=15, color='navy'),
    bgcolor='rgba(255,255,255,0.8)', bordercolor='navy', borderwidth=1, borderpad=4,
)

# ── Right: Defense PPG rank vs Win% ───────────────────────────────────────
df_def = joined[joined['def_pts_norm'].notna() & joined['win_pct'].notna()].copy()
r_def, _ = scipy_stats.pearsonr(df_def['def_pts_norm'], df_def['win_pct'])
m_def, b_def = np.polyfit(df_def['def_pts_norm'], df_def['win_pct'], 1)
x_def = np.linspace(0, 1, 300)

hover_def = (
    df_def['year'].astype(str) + ' ' + df_def['team_name'] +
    '<br>QB: ' + df_def['primary_qb'].fillna('?') +
    '<br>Def PPG Rank: ' + df_def['def_pts_rank'].astype('Int64').astype(str) +
    '/' + df_def['number_teams'].astype('Int64').astype(str) +
    '<br>Win%: ' + (df_def['win_pct'] * 100).round(1).astype(str) + '%'
)
fig_corr.add_trace(go.Scatter(
    x=df_def['def_pts_norm'], y=df_def['win_pct'],
    mode='markers',
    marker=dict(color='tomato', size=3, opacity=0.30),
    hovertext=hover_def, hoverinfo='text',
    name='All team-seasons', showlegend=False,
), row=1, col=2)
fig_corr.add_trace(go.Scatter(
    x=x_def, y=m_def * x_def + b_def,
    mode='lines', line=dict(color='darkred', width=2.5),
    name='Trend', hoverinfo='skip', showlegend=False,
), row=1, col=2)
fig_corr.add_annotation(
    row=1, col=2, x=0.05, y=0.92, xref='x domain', yref='y domain',
    text=f'<b>r = {r_def:.3f}</b>',
    showarrow=False, font=dict(size=15, color='darkred'),
    bgcolor='rgba(255,255,255,0.8)', bordercolor='darkred', borderwidth=1, borderpad=4,
)

fig_corr.update_xaxes(title_text='Composite Passer Rating', row=1, col=1)
fig_corr.update_xaxes(title_text='← Best Defense     Worst Defense →',
                       tickformat='.0%', row=1, col=2)
fig_corr.update_yaxes(title_text='Win%', tickformat='.0%', row=1, col=1)
fig_corr.update_yaxes(tickformat='.0%', row=1, col=2)
fig_corr.update_layout(
    title=dict(text='What Predicts Wins? All NFL Team-Seasons 1960–2024<br>'
                    '<sup>Each dot = one team-season (n ≈ 1,700). Trend line = OLS regression.</sup>',
               x=0.5),
    height=460, hovermode='closest',
)
fig_corr.show()

print(f'QB Rating vs Win%:           r = {r_qb:.3f}  (r² = {r_qb**2:.3f})')
print(f'Defense PPG rank vs Win%:    r = {r_def:.3f}  (r² = {r_def**2:.3f})')
print(f'|r| ratio: defense is {abs(r_def)/abs(r_qb):.1f}x more predictive than QB rating')


QB Rating vs Win%:           r = 0.532  (r² = 0.283)
Defense PPG rank vs Win%:    r = -0.700  (r² = 0.490)
|r| ratio: defense is 1.3x more predictive than QB rating


---
# Chart 5 — Expected Wins: Who Beat Their Defense? vs. Actual: Who Beat Their Defense?

**Method**: For each defensive rank tier, compute the *expected win%* from the historical average of all teams at that tier (rolling mean). Multiply by season games to get *expected wins*.

**Wins Above Expected (WAE)** = actual wins − expected wins.  
- Positive WAE → QB's team won more than the defense alone would have predicted  
- Negative WAE → underperformed the defensive advantage  

This isolates the non-defensive contribution: QB play, offense, coaching, special teams.  
Table covers all QBs with 4+ seasons as primary starter.

In [11]:
from scipy.interpolate import interp1d

# ── Build rolling-mean expected-win% model ────────────────────────────────────
_base = (joined[joined['def_pts_norm'].notna() & joined['win_pct'].notna()]
         .sort_values('def_pts_norm').reset_index(drop=True))
_w = max(60, int(len(_base) * 0.08))
_base['_exp_wp'] = _base['win_pct'].rolling(_w, center=True, min_periods=_w // 3).mean()
_valid = _base.dropna(subset=['_exp_wp'])
_exp_fn = interp1d(
    _valid['def_pts_norm'], _valid['_exp_wp'], kind='linear',
    bounds_error=False,
    fill_value=(_valid['_exp_wp'].iloc[0], _valid['_exp_wp'].iloc[-1]),
)

joined['exp_win_pct'] = joined['def_pts_norm'].apply(
    lambda x: float(_exp_fn(x)) if pd.notna(x) else np.nan
)
joined['exp_wins'] = joined['exp_win_pct'] * joined['season_games']
joined['wae']      = joined['wins'] - joined['exp_wins']

# ── Qualify: 4+ seasons as primary starter ───────────────────────────────────
_valid_rows = joined[joined['wae'].notna() & joined['wins'].notna()]
_qb_ct      = _valid_rows.groupby('primary_qb')['year'].count()
_qual       = _qb_ct[_qb_ct >= 4].index

wae_df = (
    _valid_rows[_valid_rows['primary_qb'].isin(_qual)]
    .groupby('primary_qb').agg(
        seasons      = ('year',             'count'),
        avg_wins     = ('wins',             'mean'),
        avg_exp_wins = ('exp_wins',         'mean'),
        avg_wae      = ('wae',              'mean'),
        total_wae    = ('wae',              'sum'),
        avg_rating   = ('composite_rating', 'mean'),
        avg_def_norm = ('def_pts_norm',     'mean'),
    ).reset_index().sort_values('avg_wae', ascending=False)
)

print(f'QBs with 4+ qualifying seasons: {len(wae_df)}')
print()
print('Top 15 — beats expectations most (wins above expected per season):')
top15 = wae_df.head(15)[['primary_qb','seasons','avg_wins','avg_exp_wins','avg_wae','total_wae','avg_def_norm']]
print(top15.rename(columns={'primary_qb':'QB','avg_wins':'Avg W','avg_exp_wins':'Exp W',
                             'avg_wae':'Avg WAE','total_wae':'Total WAE','avg_def_norm':'Avg Def%'})
           .round(2).to_string(index=False))
print()
print('Bottom 15 — underperforms expectations most:')
bot15 = wae_df.tail(15)[['primary_qb','seasons','avg_wins','avg_exp_wins','avg_wae','total_wae','avg_def_norm']]
print(bot15.rename(columns={'primary_qb':'QB','avg_wins':'Avg W','avg_exp_wins':'Exp W',
                             'avg_wae':'Avg WAE','total_wae':'Total WAE','avg_def_norm':'Avg Def%'})
           .round(2).to_string(index=False))


QBs with 4+ qualifying seasons: 188

Top 15 — beats expectations most (wins above expected per season):
              QB  seasons  Avg W  Exp W  Avg WAE  Total WAE  Avg Def%
     Jalen Hurts        4  12.00   8.76     3.24      12.97      0.44
 Patrick Mahomes        7  12.86   9.65     3.21      22.47      0.29
  Peyton Manning       17  11.24   8.25     2.99      50.78      0.45
      Jared Goff        8  10.25   7.84     2.41      19.26      0.53
      Drew Brees       19   9.63   7.23     2.41      45.71      0.56
     Andrew Luck        6   9.83   7.57     2.26      13.59      0.54
     Trent Green        6   8.33   6.10     2.24      13.42      0.75
Daunte Culpepper        5   7.80   5.63     2.17      10.83      0.81
  Roger Staubach        8  10.62   8.52     2.11      16.85      0.26
     Kurt Warner        9   9.11   7.02     2.09      18.85      0.61
       Tom Brady       21  12.05   9.95     2.09      43.96      0.20
       Tony Romo        8   9.75   7.81     1.94      15

In [12]:
# ── Bar chart: all 50+ QBs sorted by avg WAE ────────────────────────────────
fig_wae = go.Figure()
fig_wae.add_trace(go.Bar(
    x=wae_df['primary_qb'],
    y=wae_df['avg_wae'],
    marker_color=['#1a9850' if v >= 0 else '#d73027' for v in wae_df['avg_wae']],
    hovertemplate=(
        '<b>%{x}</b><br>' +
        'Seasons: ' + wae_df['seasons'].astype(str) + '<br>' +
        'Avg actual wins: ' + wae_df['avg_wins'].round(1).astype(str) + '<br>' +
        'Avg expected wins: ' + wae_df['avg_exp_wins'].round(1).astype(str) + '<br>' +
        'Avg WAE: <b>' + wae_df['avg_wae'].round(2).astype(str) + '</b><br>' +
        'Total WAE: ' + wae_df['total_wae'].round(1).astype(str) + '<br>' +
        'Avg QB Rating: ' + wae_df['avg_rating'].round(1).astype(str) + '<br>' +
        'Avg def percentile: ' + (wae_df['avg_def_norm'] * 100).round(0).astype(int).astype(str) + '%' +
        '<extra></extra>'
    ),
    name='Avg WAE',
))
fig_wae.add_hline(y=0, line_color='black', line_width=1.5)

# Vertical dotted lines to highlight our named QBs
named = {**GOAT_IDS, **CLIPBOARD_IDS}
named_names = set(named.values())
positions = {row.primary_qb: i for i, row in enumerate(wae_df.itertuples())}
for nm in named_names:
    if nm in positions:
        fig_wae.add_vline(x=positions[nm], line_dash='dot',
                          line_color='rgba(0,0,0,0.25)', line_width=1)

fig_wae.update_layout(
    title=dict(text='Wins Above Expected (WAE) per Season — All QBs with 4+ Starter Seasons<br>'
                    '<sup>Expected wins = historical avg wins for teams with same defensive rank. Green = beat expectations.</sup>',
               x=0.5),
    xaxis=dict(title='', tickangle=50, tickfont=dict(size=9)),
    yaxis=dict(title='Avg Wins Above Expected per Season'),
    height=500, hovermode='closest',
    bargap=0.15,
)
fig_wae.show()


In [13]:
# ── Plotly scrollable table with all QBs ─────────────────────────────────────
# Highlight rows for our named QBs
row_colors = []
for _, row in wae_df.iterrows():
    if row['primary_qb'] in named_names:
        row_colors.append('rgba(255,240,150,0.6)')  # yellow highlight
    else:
        row_colors.append('white' if _ % 2 == 0 else '#f5f5f5')

fig_tbl = go.Figure(data=[go.Table(
    columnwidth=[160, 60, 70, 70, 80, 80, 80, 80],
    header=dict(
        values=['<b>QB</b>', '<b>Seasons</b>',
                '<b>Avg Wins</b>', '<b>Avg Exp Wins</b>',
                '<b>Avg WAE</b>', '<b>Total WAE</b>',
                '<b>Avg Rating</b>', '<b>Avg Def %ile</b>'],
        fill_color='#2d6a4f',
        font=dict(color='white', size=12),
        align='center', height=28,
    ),
    cells=dict(
        values=[
            wae_df['primary_qb'],
            wae_df['seasons'],
            wae_df['avg_wins'].round(1),
            wae_df['avg_exp_wins'].round(1),
            wae_df['avg_wae'].round(2),
            wae_df['total_wae'].round(1),
            wae_df['avg_rating'].round(1),
            (wae_df['avg_def_norm'] * 100).round(0).astype(int).astype(str) + '%',
        ],
        fill_color=[row_colors] * 8,
        font=dict(size=11),
        align=['left'] + ['center'] * 7,
        height=22,
    )
)])
fig_tbl.update_layout(
    title=dict(text=f'Full WAE Rankings — {len(wae_df)} QBs with 4+ Seasons as Primary Starter<br>'
                    '<sup>Yellow rows = named QBs from our list. Sorted by Avg WAE.</sup>',
               x=0.5),
    height=min(1400, 60 + 22 * len(wae_df)),
    margin=dict(t=70, b=20, l=20, r=20),
)
fig_tbl.show()
